# ACOS — Automatic Checkout System
### Notebook 3

Runs a trained YOLO model on a checkout video, confirms products through temporal
frame-hit accumulation, and produces a priced receipt.

**Pipeline:**
```
Video frame
  → YOLO at low global floor (0.20)
  → Per-class confidence filter
  → Cross-class NMS
  → Frame-hit accumulation per class
  → Temporal confirmation (≥ N hits)
  → Receipt
```

## 1 · Install

In [27]:
!pip install -q ultralytics opencv-python numpy

## 2 · Imports

In [28]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import logging, glob, os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 3 · Price dictionary

Maps each YOLO class name to its unit price in TND.
**This is the only cell you need to edit when prices change.**

In [29]:
prices = {
    "choco_esprit_de_fete": 25, "milk_delice": 1.35, "juice_diva": 3.5,
    "soapbar_dove_shea": 4.9, "butter_jadida": 3.8, "vanillinatedsugar": 1.5,
    "disinfectant_cnett": 7.5, "cocoapowder": 2.76, "pril": 3.0,
    "conditioner_avilea": 6.0, "showergel": 5.5, "riso_scotti": 18,
    "flour": 1.4, "yogurt_danette": 0.9, "choc": 3.7,
    "pasta_spaghetti": 0.41, "rice": 1.6, "bathfoam_malizia": 10.9,
    "soapbar_dove_lavender": 3.5, "dryyeast_smartchef": 0.8,
    "butter_delice": 7.59, "judy": 6.2, "carolin": 10.69,
    "vinegar": 2.59, "dryyeast_lapatissiere": 0.9, "choco_coating": 2.3,
    "toothpaste_colgate": 3.8, "shampoo_elseve_hya": 25.59,
    "bakingpowder": 1.1, "pasta_fell": 0.41, "teabags_camomilia": 7.39,
    "chocoline": 7.29, "pasta_canelloni": 1.39, "yogurt_delice": 0.85,
    "coffee_bondin": 4.8, "chantilly_vanoise": 3.3,
    "shampoo_elseve_gly": 25.59, "shampoo_elvive": 6.8,
    "orzo": 0.41, "soap_lilas": 10.2, "mustard": 6.5,
    "harissa": 2.2, "liquidsoap_naya": 3.8,
    "chocospread_vanoise": 8.7, "canned_peas": 1.9
}
print(f"{len(prices)} products in price list")

45 products in price list


## 4 · Cross-class NMS

Standard YOLO NMS only suppresses overlapping boxes of the **same** class.
`cross_class_nms` also handles **different-class** overlaps — catching cases where
the model fires two labels on the same physical object (e.g. `dryyeast` partially
misidentified as `vanillinatedsugar`).

**Why `iou_threshold=0.40` and not 0.20?**
At 1920×1080, large product boxes naturally share 20–40% overlap just from being
placed close together. A threshold of 0.20 suppressed 37–60% of valid detections
in testing. At 0.40, only boxes that genuinely cover the same object are removed.

In [30]:
def _iou(a, b):
    """Intersection over Union between two [x1,y1,x2,y2] boxes."""
    ix1 = max(a[0], b[0])
    iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2])
    iy2 = min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-6)


def cross_class_nms(detections, iou_threshold=0.40):
    """
    Suppress overlapping boxes across ALL classes (not just same-class).

    Args:
        detections:    list of [x1, y1, x2, y2, conf, cls]
        iou_threshold: boxes with IoU above this value vs a higher-conf box
                       are suppressed. 0.40 is calibrated for 1920x1080.
    Returns:
        Filtered list — at most one detection per spatial region.
    """
    if not detections:
        return []
    dets = sorted(detections, key=lambda d: d[4], reverse=True)
    kept = []
    while dets:
        best = dets.pop(0)
        kept.append(best)
        dets = [d for d in dets if _iou(best[:4], d[:4]) < iou_threshold]
    return kept

print("cross_class_nms defined")

cross_class_nms defined


## 5 · Checkout system — `StaticSceneCheckout`

### How confirmation works
`class_conf_history` is a dict that accumulates the best confidence seen per class
**across the entire video**. It is never reset. A product is confirmed when its
accumulated frame-hit count reaches `min_confirm_frames`. Products that leave the
frame before the video ends are still counted correctly.

### Why per-class configuration is necessary
A single global threshold assumes the model learned all classes equally — it never does.

| Parameter | Purpose |
|---|---|
| `detection_floor=0.20` | YOLO is called at this low floor so weak detectors reach the per-class gate. Without this, `class_conf_overrides` below 0.40 would never see any candidates. |
| `class_conf_overrides` | Per-class minimum confidence. Use for classes the model fires weakly on (e.g. `soapbar_dove_shea` fires correctly at 0.25–0.35). |
| `min_frames_overrides` | Per-class frame threshold. A product visible for 8 frames and one visible for 130 frames cannot share the same confirmation threshold. |
| `multi_instance=True` | Uses `class_max_simultaneous` — the peak number of boxes of that class seen in a single frame. Correct general signal for quantity without hardcoding counts. |
| `qty_overrides` | Force a fixed quantity. Only use this as a last resort when `multi_instance` cannot distinguish counts (e.g. identical products stacked on top of each other). |

In [31]:
class StaticSceneCheckout:
    """
    Checkout for pan-camera / static-object videos.

    Pipeline per frame:
      1. YOLO at low global floor (detection_floor)
      2. Per-class confidence filter
      3. Cross-class NMS
      4. Accumulate per-class frame-hit counts (never reset)
      5. Confirm when hits >= min_confirm_frames (per-class override supported)
    """

    def __init__(self,
                 model_path,
                 price_dict,
                 conf_threshold=0.40,
                 detection_floor=0.20,
                 min_confirm_frames=6,
                 nms_iou_threshold=0.40,
                 max_box_ratio=0.92,
                 multi_instance=False,
                 class_conf_overrides=None,
                 min_frames_overrides=None,
                 qty_overrides=None,
                 debug=False):
        self.model = YOLO(model_path)
        self.prices = price_dict
        self.conf_threshold = conf_threshold
        self.detection_floor = detection_floor
        self.min_confirm_frames = min_confirm_frames
        self.nms_iou_threshold = nms_iou_threshold
        self.max_box_ratio = max_box_ratio
        self.multi_instance = multi_instance
        self.class_conf_overrides = class_conf_overrides or {}
        self.min_frames_overrides = min_frames_overrides or {}
        self.qty_overrides = qty_overrides or {}
        self.debug = debug

        # Accumulates across entire video — never reset
        self.class_conf_history = defaultdict(list)
        self.class_max_simultaneous = defaultdict(int)

        self.frame_count = 0
        self.raw_count = 0
        self.post_nms_count = 0
        self.suppressed_by_nms = 0

    def _get_class_conf_threshold(self, label):
        """Per-class conf threshold, falling back to global default."""
        return self.class_conf_overrides.get(label, self.conf_threshold)

    def _get_min_frames(self, label):
        """Per-class min_confirm_frames, falling back to global default."""
        return self.min_frames_overrides.get(label, self.min_confirm_frames)

    def process_frame(self, frame):
        """Run full pipeline on one frame. Returns annotated frame."""
        self.frame_count += 1
        h, w = frame.shape[:2]
        max_area = h * w * self.max_box_ratio

        # Step 1: YOLO at low global floor
        results = self.model(frame, conf=self.detection_floor, verbose=False)[0]

        # Step 2: per-class confidence + area filter
        raw = []
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            label = self.model.names[cls]
            area = (x2 - x1) * (y2 - y1)
            min_conf = self._get_class_conf_threshold(label)
            if conf >= min_conf and area <= max_area:
                raw.append([x1, y1, x2, y2, conf, cls])

        self.raw_count += len(raw)

        # Step 3: cross-class NMS
        clean = cross_class_nms(raw, iou_threshold=self.nms_iou_threshold)
        self.post_nms_count += len(clean)
        self.suppressed_by_nms += len(raw) - len(clean)

        # Step 4: accumulate per-class stats for this frame
        frame_cls_conf = defaultdict(float)
        frame_cls_count = defaultdict(int)
        for det in clean:
            cls = int(det[5])
            conf = det[4]
            frame_cls_count[cls] += 1
            if conf > frame_cls_conf[cls]:
                frame_cls_conf[cls] = conf

        for cls, best_conf in frame_cls_conf.items():
            self.class_conf_history[cls].append(best_conf)
        for cls, cnt in frame_cls_count.items():
            if cnt > self.class_max_simultaneous[cls]:
                self.class_max_simultaneous[cls] = cnt

        if self.debug and frame_cls_conf:
            logger.info(f"Frame {self.frame_count}: "
                        f"{[self.model.names[c] for c in frame_cls_conf]}")

        return self._draw_frame(frame, raw, clean)

    def _draw_frame(self, frame, raw_dets, clean_dets):
        """Annotate frame. Suppressed boxes shown in dim blue."""
        h, w = frame.shape[:2]
        clean_coords = {(round(d[0]), round(d[1]), round(d[2]), round(d[3]))
                        for d in clean_dets}

        # Suppressed boxes (dim blue, thin)
        for d in raw_dets:
            key = (round(d[0]), round(d[1]), round(d[2]), round(d[3]))
            if key not in clean_coords:
                cv2.rectangle(frame, (int(d[0]), int(d[1])), (int(d[2]), int(d[3])),
                              (40, 40, 160), 1)

        # Accepted boxes (green)
        for d in clean_dets:
            x1, y1, x2, y2 = map(int, d[:4])
            label = self.model.names[int(d[5])]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 215, 0), 2)
            cv2.putText(frame, f"{label} {d[4]:.2f}",
                        (x1, max(y1 - 8, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, (0, 215, 0), 2)

        # HUD overlay
        confirmed = self._get_confirmed()
        y = 30
        cv2.putText(frame, "=== CHECKOUT v5 ===",
                    (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 230, 0), 2)
        y += 26
        cv2.putText(frame, f"frame {self.frame_count}  NMS suppressed: {self.suppressed_by_nms}",
                    (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (160, 160, 160), 1)
        y += 20

        running_total = 0.0
        for label, qty in sorted(confirmed.items()):
            unit = self.prices.get(label, 0)
            sub = unit * qty
            running_total += sub
            cv2.putText(frame, f"{label}  x{qty}  {sub:.2f} TND",
                        (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.46, (255, 230, 0), 1)
            y += 19

        cv2.rectangle(frame, (5, h - 38), (w - 5, h - 5), (0, 220, 220), 2)
        cv2.putText(frame,
                    f"TOTAL: {running_total:.2f} TND  |  {len(confirmed)} products",
                    (10, h - 14), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (0, 220, 220), 2)
        return frame

    def _get_confirmed(self):
        """
        Returns confirmed product dict.
        Reads from class_conf_history which accumulates across the whole
        video and is never reset — products that have left the frame are
        still counted correctly at receipt time.
        """
        confirmed = {}
        for cls, conf_list in self.class_conf_history.items():
            label = self.model.names[cls]
            threshold = self._get_min_frames(label)
            if len(conf_list) >= threshold:
                if label in self.qty_overrides:
                    qty = self.qty_overrides[label]
                elif self.multi_instance:
                    qty = self.class_max_simultaneous[cls]
                else:
                    qty = 1
                confirmed[label] = qty
        return confirmed

    def get_receipt(self):
        """Print final receipt and per-class diagnostics."""
        confirmed = self._get_confirmed()
        total = sum(self.prices.get(lbl, 0) * qty for lbl, qty in confirmed.items())

        print("\n" + "=" * 54)
        print("          FINAL RECEIPT  —  CHECKOUT v5")
        print("=" * 54)
        for label, qty in sorted(confirmed.items()):
            unit = self.prices.get(label, 0)
            sub = unit * qty
            print(f"  {label:40s} x{qty}  {sub:6.2f} TND")
        print("-" * 54)
        print(f"  {'TOTAL':40s}     {total:6.2f} TND")
        print("=" * 54)

        print("\n  Per-class frame-hit counts (sorted by hits):")
        for cls, conf_list in sorted(self.class_conf_history.items(),
                                     key=lambda x: -len(x[1])):
            label = self.model.names[cls]
            avg_c = float(np.mean(conf_list)) if conf_list else 0.0
            min_f = self._get_min_frames(label)
            min_c = self._get_class_conf_threshold(label)
            status = "OK" if len(conf_list) >= min_f else "X  <- needs tuning"
            print(f"  [{status}] {label:40s} frames={len(conf_list):4d}  "
                  f"avg_conf={avg_c:.2f}  (min_conf={min_c:.2f}, min_frames={min_f})")

        print(f"\n  Pipeline stats:")
        print(f"    Total frames      : {self.frame_count}")
        print(f"    Raw detections    : {self.raw_count}")
        print(f"    Suppressed by NMS : {self.suppressed_by_nms}"
              f"  ({100*self.suppressed_by_nms/max(self.raw_count,1):.1f}%)")
        print(f"    After NMS         : {self.post_nms_count}")
        print(f"    Confirmed         : {len(confirmed)}")
        print("=" * 54 + "\n")
        return confirmed, total

print("StaticSceneCheckout defined")

StaticSceneCheckout defined


## 6 · Main function

Wraps the checkout system in a video I/O loop.

### Tuning guide

| Parameter | Default | Guidance |
|---|---|---|
| `conf_threshold` | 0.80 | Global gate for well-detected classes. Do not lower this globally — use `class_conf_overrides` for specific weak classes instead. |
| `detection_floor` | 0.20 | YOLO call floor. **Never raise this** — it must stay below all `class_conf_overrides` values. |
| `min_confirm_frames` | 10 | Frames needed to confirm a product. Raise for longer videos, lower only if products are very briefly visible. |
| `nms_iou_threshold` | 0.40 | Cross-class NMS threshold for 1080p. Lower to 0.30 only if two labels persistently fire on the same object. |
| `multi_instance` | True | Counts peak simultaneous same-class boxes as quantity. Correct general approach for multiple identical products. |
| `class_conf_overrides` | see run cell | Per-class confidence gate. Use `debug=True` to find which classes need lowering. |
| `min_frames_overrides` | see run cell | Per-class frame threshold. Lower for products only briefly visible in the video. |

In [32]:
def main(video_path,
         model_path,
         output_path="output_v5.mp4",
         conf_threshold=0.40,
         detection_floor=0.20,
         min_confirm_frames=6,
         nms_iou_threshold=0.40,
         multi_instance=False,
         class_conf_overrides=None,
         min_frames_overrides=None,
         qty_overrides=None,
         debug=False):

    checkout = StaticSceneCheckout(
        model_path=model_path,
        price_dict=prices,
        conf_threshold=conf_threshold,
        detection_floor=detection_floor,
        min_confirm_frames=min_confirm_frames,
        nms_iou_threshold=nms_iou_threshold,
        max_box_ratio=0.92,
        multi_instance=multi_instance,
        class_conf_overrides=class_conf_overrides,
        min_frames_overrides=min_frames_overrides,
        qty_overrides=qty_overrides,
        debug=debug
    )

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        logger.error(f"Cannot open: {video_path}")
        return None

    W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(output_path,
                          cv2.VideoWriter_fourcc(*"mp4v"),
                          fps, (W, H))

    logger.info(f"Video      : {video_path}")
    logger.info(f"Resolution : {W}x{H}  FPS={fps:.0f}  frames={total}")
    logger.info(f"Config     : conf>={conf_threshold}  floor={detection_floor}  "
                f"min_frames={min_confirm_frames}  nms={nms_iou_threshold}")

    n = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        n += 1
        out.write(checkout.process_frame(frame))
        if n % 60 == 0:
            logger.info(f"  {n}/{total} ({100*n/max(total,1):.0f}%)")

    cap.release()
    out.release()
    logger.info(f"Output: {output_path}")
    checkout.get_receipt()
    return checkout

print("main() defined")

main() defined


## 7 · Run

> **Expected runtime:** ~10–20 seconds on Kaggle T4 GPU · ~5–25 minutes on CPU only.
> Check GPU is enabled: Settings → Accelerator → GPU T4 x1.

**Diagnosing missed products using the receipt output:**
- `[X] frames=N` where N < `min_frames` → lower `min_frames_overrides` for that class
- `avg_conf` consistently below 0.40 → add a `class_conf_overrides` entry for that class
- Product appears as false positive → raise its `class_conf_overrides` value

In [33]:
# ── GPU check ─────────────────────────────────────────────────────────────
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: running on CPU — inference will be slow")

# ── Paths ──────────────────────────────────────────────────────────────────
MODEL_PATH  = "/kaggle/input/datasets/mayaralabidi/checkout-model-yolov8s/runs/detect/train/weights/best.pt"
VIDEO_PATH  = "/kaggle/input/datasets/mayaralabidi/demovid2/VID_demo.mp4"
OUTPUT_PATH = "output_v5.mp4"

# Auto-discovery if exact paths don't exist
if not os.path.exists(MODEL_PATH):
    found = glob.glob("/kaggle/input/**/best.pt", recursive=True)
    if found:
        MODEL_PATH = found[0]
        print(f"Model auto-discovered: {MODEL_PATH}")
    else:
        raise FileNotFoundError("No model found. Check MODEL_PATH.")

if not os.path.exists(VIDEO_PATH):
    for ext in ["*.mp4", "*.avi", "*.mov", "*.mkv"]:
        found = glob.glob(f"/kaggle/input/**/{ext}", recursive=True)
        if found:
            VIDEO_PATH = found[0]
            print(f"Video auto-discovered: {VIDEO_PATH}")
            break
    else:
        raise FileNotFoundError("No video found. Check VIDEO_PATH.")

print(f"Model: {'OK' if os.path.exists(MODEL_PATH) else 'NOT FOUND'} — {MODEL_PATH}")
print(f"Video: {'OK' if os.path.exists(VIDEO_PATH) else 'NOT FOUND'} — {VIDEO_PATH}")

# ── Run ────────────────────────────────────────────────────────────────────
checkout = main(
    video_path=VIDEO_PATH,
    model_path=MODEL_PATH,
    output_path=OUTPUT_PATH,

    # Global defaults
    conf_threshold=0.80,          # strict global gate — weak classes use overrides below
    detection_floor=0.20,         # YOLO call floor — must stay low for overrides to work
    min_confirm_frames=10,        # product must appear in >=10 frames to be confirmed
    nms_iou_threshold=0.40,       # calibrated for 1920x1080

    # Per-class confidence overrides
    # These classes fire correctly at lower confidence — verified with debug=True
    class_conf_overrides={
        "soapbar_dove_shea":     0.25,
        "soapbar_dove_lavender": 0.25,
        "toothpaste_colgate":    0.30,
    },

    # Per-class frame threshold overrides
    # These products are briefly visible in the video — lower their confirmation bar
    min_frames_overrides={
        "soapbar_dove_shea":     5,
        "soapbar_dove_lavender": 5,
        "toothpaste_colgate":    5,
    },

    multi_instance=True,          # counts simultaneous same-class boxes as quantity
    qty_overrides={},             # leave empty — let multi_instance handle counting
    debug=False,                  # set True on first run to diagnose missed products
)

GPU available: False
Model: OK — /kaggle/input/datasets/mayaralabidi/checkout-model-yolov8s/runs/detect/train/weights/best.pt
Video: OK — /kaggle/input/datasets/mayaralabidi/demovid2/VID_demo.mp4


INFO:__main__:Video      : /kaggle/input/datasets/mayaralabidi/demovid2/VID_demo.mp4
INFO:__main__:Resolution : 1920x1080  FPS=30  frames=325
INFO:__main__:Config     : conf>=0.8  floor=0.2  min_frames=10  nms=0.4
INFO:__main__:  60/325 (18%)
INFO:__main__:  120/325 (37%)
INFO:__main__:  180/325 (55%)
INFO:__main__:  240/325 (74%)
INFO:__main__:  300/325 (92%)
INFO:__main__:Output: output_v5.mp4



          FINAL RECEIPT  —  CHECKOUT v5
  canned_peas                              x1    1.90 TND
  chantilly_vanoise                        x2    6.60 TND
  dryyeast_smartchef                       x1    0.80 TND
  riso_scotti                              x1   18.00 TND
  shampoo_elseve_gly                       x1   25.59 TND
  soapbar_dove_lavender                    x1    3.50 TND
  soapbar_dove_shea                        x1    4.90 TND
  toothpaste_colgate                       x1    3.80 TND
------------------------------------------------------
  TOTAL                                         65.09 TND

  Per-class frame-hit counts (sorted by hits):
  [OK] chantilly_vanoise                        frames= 113  avg_conf=0.94  (min_conf=0.80, min_frames=10)
  [OK] shampoo_elseve_gly                       frames= 103  avg_conf=0.91  (min_conf=0.80, min_frames=10)
  [OK] dryyeast_smartchef                       frames=  56  avg_conf=0.88  (min_conf=0.80, min_frames=10)
  [OK] riso_s